# Chapter 2 — Build a billing-support copilot with DSPy

This notebook uses **one realistic task from start to finish**: applying a SaaS billing policy to customer tickets. We will grow one small program into a system someone could actually ship:

1. `Predict` establishes a measurable baseline.
2. `ChainOfThought` improves multi-rule policy decisions without changing the task contract.
3. `BootstrapFewShot` compiles successful traces into the program.
4. `ReAct` looks up account facts and calls an exact calculator when the ticket alone is insufficient.
5. A custom `dspy.Module` composes three Signatures: a router, a policy specialist, and the tool-using agent.
6. `RLM` resolves the same kind of case from hundreds of account events without stuffing the whole history into one prompt.

The structure follows the useful idea in [DSPy's interactive homepage example](https://dspy.ai/): keep the task recognizable, then make small changes that add a real capability.

**Requirements:** Python 3.12+, an `OPENROUTER_API_KEY`, and Deno for the final RLM section. The defaults deliberately use a small 8B model for the deployed program and a stronger model only for routing, teaching, and RLM control. Set `DSPY_SMALL_MODEL` or `DSPY_STRONG_MODEL` to try other models without changing the program.

## How the chapter and notebook fit together

The printed chapter uses deliberately small snippets so each DSPy abstraction is easy to see. The simplified examples appear beside the corresponding full notebook sections as syntax-highlighted reference cells. They are intentionally non-executing, so they cannot reconfigure your model, repeat paid calls, or replace variables used by the notebook.

The executable cells remain one richer billing-support example that runs end to end. When the chapter introduces a compact `BillingDecision`, the nearby notebook section expands the same idea into `ResolveBillingCase`; when the chapter shows a minimal metric or optimizer, the executable path applies that abstraction to the full decision, amount, and routing contract.

In [ ]:
# Install the repository's complete, pinned environment.
from pathlib import Path

requirements_path = next(
    (
        directory / "requirements.txt"
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / "requirements.txt").is_file()
    ),
    None,
)
if requirements_path is None:
    raise FileNotFoundError("Could not find the repository requirements.txt from this working directory.")

%pip install -r {requirements_path} -q

## 1. Configure language models

DSPy's LM abstraction keeps the rest of the code provider-independent. The signatures, modules, tools, metrics, and dataset below do not know which model serves them.

We use the smaller model for most calls because that is the economically interesting production setup. It is intentionally imperfect: the point is to improve the program, not hide every problem behind a frontier model.

### Configure an OpenAI language model

The executable setup below uses the notebook's OpenRouter defaults, while the Signature and Module APIs stay the same.

```python
import dspy
import os

# Setting up an OpenAI model
lm = dspy.LM('openai/gpt-5.4-mini',
    api_key=os.getenv('OPENAI_API_KEY'), # load from env
    temperature=1, max_tokens=16000) # additional params

# Configure globally
dspy.configure(lm=lm)
```

### Call the configured LM directly

```python
lm("Hello")
# >> ['Hi — how can I help you today?']
```

### Switch model providers

```python
# Setting up an Anthropic model
claude_lm = dspy.LM('anthropic/claude-haiku-4-5-20251001',
    api_key=os.getenv('ANTHROPIC_API_KEY')) # load from env

# Setting up a Google model
gem_lm = dspy.LM('gemini/gemini-3.1-flash-lite',
    api_key=os.getenv('GEMINI_API_KEY')) # load from env

# Setting up a small open source model
small_lm = dspy.LM(
    "openrouter/meta-llama/llama-3.1-8b-instruct",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.0,
    max_tokens=1000,
)
```

In [ ]:
import os
import shutil
from typing import Literal

import dspy
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError(
        "Set OPENROUTER_API_KEY in your environment or .env file before running this notebook."
    )

SMALL_MODEL = os.getenv(
    "DSPY_SMALL_MODEL",
    "openrouter/meta-llama/llama-3.1-8b-instruct",
)
STRONG_MODEL = os.getenv(
    "DSPY_STRONG_MODEL",
    "openrouter/google/gemini-3.1-flash-lite",
)

small_lm = dspy.LM(
    SMALL_MODEL,
    temperature=0.0,
    max_tokens=1000,
    cache=True,
    timeout=60,
    num_retries=2,
)
strong_lm = dspy.LM(
    STRONG_MODEL,
    temperature=0.0,
    max_tokens=2000,
    cache=True,
    timeout=60,
    num_retries=2,
)

dspy.configure(lm=small_lm)

print("DSPy:", dspy.__version__)
print("Deployed model:", SMALL_MODEL)
print("Strong model:", STRONG_MODEL)

## 2. Define the task once with a Signature

Nimbus Cloud's support team needs to make three coupled decisions: what action is allowed, the exact dollar amount, and which team owns the case. A plausible-sounding reply is not enough if any of those fields is wrong.

The Signature is the contract. Notice that it says *what* must be produced, but not whether DSPy should answer directly, reason first, call tools, or run code. We will change those execution strategies while keeping these outputs stable.

### Define the smallest Signature

```python
class BillingDecision(dspy.Signature):
    """Decide whether to approve, deny, or escalate a billing request."""

    request: str = dspy.InputField()
    decision: str = dspy.OutputField()
```

### Add policy context, descriptions, and a constrained output

```python
from typing import Literal

class BillingDecision(dspy.Signature):
    """Apply the billing policy to a customer request."""

    request: str = dspy.InputField(
        desc="The customer’s billing request"
    )
    policy: str = dspy.InputField(
        desc="The billing policy to apply"
    )
    decision: Literal["approve", "deny", "escalate"] = dspy.OutputField(
        desc="How the request should be handled"
    )
```

In [ ]:
BILLING_POLICY = """
Nimbus Cloud billing policy (all day counts are inclusive):
1. Duplicate charges: approve a refund of the duplicated charge only. Route to billing.
2. Annual renewals:
   - Days 0-14 after renewal: approve the full renewal only when post-renewal usage is at most 10 compute-hours.
   - Days 15-30: approve 50% only when post-renewal usage is exactly 0 compute-hours.
   - Otherwise deny. Route all renewal cases to retention.
3. Monthly plan cancellations: deny the current month; cancellation takes effect next cycle. Route to retention.
4. Verified platform outage: route to reliability and issue a service credit, not a card refund. Credit is
   min(monthly_fee * downtime_minutes / 43,200 * 10, monthly_fee * 0.30), rounded to cents.
   Only approve when monthly uptime is below 99.9%; otherwise deny with $0.
5. Any suspected account takeover or unrecognized account changes: escalate to security, amount $0,
   and do not promise a refund.
If facts needed by a rule conflict or are missing, escalate to billing with amount $0.
""".strip()


class ResolveBillingCase(dspy.Signature):
    """Apply the billing policy to a customer request. Never invent missing facts."""

    request: str = dspy.InputField()
    account_snapshot: str = dspy.InputField()
    policy: str = dspy.InputField()
    decision: Literal["approve", "deny", "escalate"] = dspy.OutputField()
    amount: float = dspy.OutputField(
        desc="Approved refund or credit in USD; 0 for deny or escalate"
    )
    route: Literal["billing", "retention", "reliability", "security"] = dspy.OutputField()
    reply: str = dspy.OutputField(desc="A short customer-facing explanation")

## 3. Turn requirements into Examples and a metric

An evaluation case contains the program inputs plus the expected operational outputs. `.with_inputs(...)` tells DSPy which fields are inputs; the remaining fields are labels.

The metric awards one third each for the decision, dollar amount, and route. That gives us useful partial credit during development. When an optimizer passes a `trace`, the same metric becomes strict: only a completely correct case counts as a successful demonstration.

### Build Examples directly

```python
renewal_policy = (
    "For annual renewals, approve within 14 days only when usage is no more "
    "than 10 compute-hours. From days 15–30, approve only when usage is "
    "exactly zero. Otherwise, deny."
)

examples = [
    dspy.Example(
        # Inputs
        request=(
            "Please refund my annual renewal. It renewed 9 days ago, "
            "and I have used 14 compute-hours."
        ),
        policy=renewal_policy,
        # Expected output
        decision="deny",
    ).with_inputs("request", "policy"),

    dspy.Example(
        request=(
            "We forgot to cancel our annual plan. It renewed 18 days ago, "
            "and we have not used it since."
        ),
        policy=renewal_policy,
        decision="approve",
    ).with_inputs("request", "policy"),
]
```

### Convert dictionaries into Examples

```python
dataset = [
    {
        "request": (
            "Please refund my annual renewal. It renewed 9 days ago, "
            "and I have used 14 compute-hours."
        ),
        "policy": renewal_policy,
        "decision": "deny",
    },
    {
        "request": (
            "We forgot to cancel our annual plan. It renewed 18 days ago, "
            "and we have not used it since."
        ),
        "policy": renewal_policy,
        "decision": "approve",
    },
]

examples = [
    dspy.Example(**item).with_inputs("request", "policy")
    for item in dataset
]
```

### Define an exact-match metric

```python
def decision_match(example, prediction, trace=None):
    """Check whether the predicted decision matches the expected decision."""
    return example.decision == prediction.decision
```

### Run a manual evaluation loop

```python
program = dspy.Predict(BillingDecision)

scores = []

for example in examples:
    prediction = program(**example.inputs())
    score = decision_match(example, prediction)
    scores.append(score)

average_score = sum(scores) / len(scores)
print(f"Average score: {average_score:.2f}")
```

### Evaluate with DSPy's utility

```python
# Create evaluator
evaluator = dspy.Evaluate(
    devset=examples, # your dataset
    metric=decision_match, # your eval metric
    num_threads=2, # how many to run at a time
    display_progress=True,
    display_table=2,
)

# Run the evaluation
evaluation = evaluator(program)
print(f"Accuracy: {evaluation.score}%")
```

### Create train, validation, and test splits

```python
import random

# Make the split reproducible
random.seed(42)
random.shuffle(examples)

n = len(examples)

# Optionally reserve 20% for final testing
n_test = int(0.2 * n)
testset = examples[:n_test]
optimization_data = examples[n_test:]

# Split the remaining data between training and validation
val_fraction = 0.5 if n < 200 else 0.2
n_val = int(len(optimization_data) * val_fraction)

valset = optimization_data[:n_val]
trainset = optimization_data[n_val:]

print("Train set:", len(trainset))
print("Val set:", len(valset))
print("Test set:", len(testset))

# With 100 examples:
# >> Train set: 40
# >> Val set: 40
# >> Test set: 20
```

In [ ]:
CASES = [
    (
        "Please refund my annual renewal.",
        "Renewed 9 days ago for $1,200; 14 compute-hours since renewal.",
        "deny", 0.0, "retention",
    ),
    (
        "We forgot to cancel before annual renewal.",
        "Renewed 18 days ago for $800; 0 compute-hours since renewal.",
        "approve", 400.0, "retention",
    ),
    (
        "The platform was down for 216 minutes. Credit us.",
        "Monthly fee $1,000; measured uptime 99.5%; verified outage 216 minutes.",
        "approve", 50.0, "reliability",
    ),
    (
        "The platform was down for 100 minutes. Credit us.",
        "Monthly fee $1,299; measured uptime 99.7%; verified outage 100 minutes.",
        "approve", 30.07, "reliability",
    ),
    (
        "Someone changed our payout account and I don't recognize it.",
        "Owner reports an unrecognized payout-account change 20 minutes ago.",
        "escalate", 0.0, "security",
    ),
]

dataset = [
    dspy.Example(
        request=request,
        account_snapshot=snapshot,
        policy=BILLING_POLICY,
        decision=decision,
        amount=amount,
        route=route,
    ).with_inputs("request", "account_snapshot", "policy")
    for request, snapshot, decision, amount, route in CASES
]


def billing_metric(example, pred, trace=None):
    checks = [
        pred.decision == example.decision,
        abs(float(pred.amount) - example.amount) < 0.01,
        pred.route == example.route,
    ]
    score = sum(checks) / len(checks)
    return score == 1.0 if trace is not None else score


def evaluate_cases(program, examples, label):
    rows = []
    predictions = []
    errors = []
    for index, example in enumerate(examples):
        try:
            pred = program(**example.inputs())
            score = billing_metric(example, pred)
            predictions.append(pred)
            rows.append({
                "case": index,
                "expected": f"{example.decision} ${example.amount:.2f} → {example.route}",
                "predicted": f"{pred.decision} ${float(pred.amount):.2f} → {pred.route}",
                "score": score,
            })
        except Exception as exc:
            predictions.append(None)
            errors.append((index, exc))
            rows.append({
                "case": index,
                "expected": f"{example.decision} ${example.amount:.2f} → {example.route}",
                "predicted": f"ERROR: {type(exc).__name__}",
                "score": 0.0,
            })
    frame = pd.DataFrame(rows)
    mean_score = frame["score"].mean()
    print(f"{label}: {mean_score:.3f}")
    print(frame.to_string(index=False))
    if errors:
        details = "; ".join(
            f"case {index}: {type(exc).__name__}: {exc}" for index, exc in errors
        )
        raise RuntimeError(f"{label} failed to produce predictions: {details}") from errors[0][1]
    return mean_score, predictions

## 4. Baseline: direct prediction

`dspy.Predict` is the smallest useful DSPy program. On this task, the small model tends to jump to a familiar policy clause or lose track of the calculation. That is exactly what a baseline should reveal.

### Use the Predict Module

```python
module = dspy.Predict(BillingDecision)

result = module(
    request="I renewed eight days ago and have used four compute-hours.",
    policy="Approve annual-renewal refunds within 14 days when usage is no more than 10 compute-hours.",
)

print(result.decision)

# >> approve
```

In [ ]:
direct = dspy.Predict(ResolveBillingCase)
baseline_score, _ = evaluate_cases(direct, dataset, "Predict")

## 5. Same Signature, better strategy: ChainOfThought

We do not add a hand-written mega-prompt or change the expected output. We swap one module. `ChainOfThought` adds a reasoning field before the operational fields, which gives this small model room to identify the rule, check its conditions, and do the arithmetic before committing to an answer.

### Swap in ChainOfThought

```python
cot_module = dspy.ChainOfThought(BillingDecision)

cot_result = cot_module(
    request="I renewed 18 days ago and have used four compute-hours.",
    policy=(
        "Approve annual-renewal refunds within 14 days when usage is no more "
        "than 10 compute-hours. From days 15–30, approve only when usage is "
        "exactly zero. Otherwise, deny."
    ),
)

print(cot_result.reasoning)
print(cot_result.decision)

# >> The request is in the 15–30 day window, but usage is not zero.
# >> deny
```

In [ ]:
reasoned = dspy.ChainOfThought(ResolveBillingCase)
cot_score, cot_predictions = evaluate_cases(reasoned, dataset, "ChainOfThought")

assert cot_score > baseline_score, (
    "This example is intended to demonstrate a real ChainOfThought improvement. "
    "If your override model changes that result, inspect the per-case table and choose a smaller deployed model."
)
print(f"Improvement: {baseline_score:.3f} → {cot_score:.3f}")

In [ ]:
# Inspect one decision without changing the public Signature.
worked_example = cot_predictions[2]
print(worked_example.reasoning)
print("Final:", worked_example.decision, worked_example.amount, worked_example.route)

## 6. Compile the program with an Optimizer

Reasoning helps, but we still have repeated failure patterns. `BootstrapFewShot` runs a teacher on labeled training cases, keeps only traces that pass the strict form of our metric, and installs those traces as demonstrations in the student program.

This is a deliberately small smoke-test budget. Chapter 3 builds a proper train/validation/test workflow; here we only need to see the optimizer improve unseen cases while the deployed LM remains the same 8B model.

### Compile with GEPA

The executable notebook uses `BootstrapFewShot` for a fast smoke test; the chapter introduces the fuller GEPA train/validation workflow.

```python
# Use a stronger model to analyze failures and propose new instructions
strong_lm = dspy.LM(
    "openrouter/google/gemini-3.1-flash-lite",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

optimizer = dspy.GEPA(
    metric=decision_match,
    reflection_lm=strong_lm,
    max_full_evals=3,
)

optimized_program = optimizer.compile(
    program,
    trainset=trainset,
    valset=valset,
)

result = optimized_program(
    request=(
        "Please refund my annual renewal. It renewed 9 days ago, "
        "and I have used 14 compute-hours."
    ),
    policy=renewal_policy,
)

print(result.decision)
```

In [ ]:
OPTIMIZER_POLICY = """
Nimbus Cloud billing policy:
1. Annual renewals: days 0-14 approve the full renewal only when usage is at most 10 compute-hours;
days 15-30 approve 50% only when usage is exactly 0; otherwise deny. Route renewals to retention.
2. Verified outage: when uptime is below 99.9%, approve a reliability credit of
min(monthly_fee * downtime_minutes / 43,200 * 10, monthly_fee * 0.30), rounded to cents.
3. Suspected account takeover or unrecognized account changes: escalate to security for $0.
Missing facts means escalate to billing for $0.
""".strip()


def make_example(request, snapshot, decision, amount, route):
    return dspy.Example(
        request=request,
        account_snapshot=snapshot,
        policy=OPTIMIZER_POLICY,
        decision=decision,
        amount=amount,
        route=route,
    ).with_inputs("request", "account_snapshot", "policy")


optimizer_trainset = [
    make_example("Refund my annual renewal.", "Renewed 9 days ago for $1,200; used 14 hours.", "deny", 0.0, "retention"),
    make_example("We forgot to cancel.", "Renewed 18 days ago for $800; used 0 hours.", "approve", 400.0, "retention"),
    make_example("Credit the outage.", "Fee $1,000; uptime 99.5%; verified outage 216 minutes.", "approve", 50.0, "reliability"),
    make_example("I do not recognize the payout change.", "Owner reports an unrecognized payout change.", "escalate", 0.0, "security"),
]
optimizer_devset = [
    make_example("Refund my renewal.", "Renewed 20 days ago for $600; used 0 hours.", "approve", 300.0, "retention"),
    make_example("Refund my renewal.", "Renewed 8 days ago for $1,200; used 4 hours.", "approve", 1200.0, "retention"),
]

student = dspy.ChainOfThought(ResolveBillingCase)
before_opt_score, _ = evaluate_cases(student, optimizer_devset, "Before optimization")

teacher = dspy.ChainOfThought(ResolveBillingCase)
teacher.set_lm(strong_lm)

optimizer = dspy.BootstrapFewShot(
    metric=billing_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=0,
    max_rounds=1,
)
optimized = optimizer.compile(
    student=student,
    teacher=teacher,
    trainset=optimizer_trainset,
)

optimized_score, optimized_predictions = evaluate_cases(
    optimized,
    optimizer_devset,
    "After optimization",
)
print("Bootstrapped demonstrations:", len(optimized.predict.demos))
assert optimized_score > before_opt_score
assert all(
    billing_metric(example, prediction, trace=True)
    for example, prediction in zip(optimizer_devset, optimized_predictions)
), "Every optimized dev prediction must satisfy the strict metric."

## 7. Add a capability: ReAct can retrieve facts and calculate exactly

The programs above can only use the snapshot placed in their prompt. A live ticket usually contains a customer ID, not a trustworthy account dump. No amount of chain-of-thought can retrieve a fee or outage record the model has never seen.

`dspy.ReAct` adds a reason–act–observe loop. The tools below are in-memory fixtures so the notebook is reproducible, but their interfaces are shaped like real internal service calls. The agent gets a narrowly retrieved outage-policy section instead of the whole handbook.

### Give a ReAct agent an account lookup tool

```python
accounts = {
    "C-204": "Verified outage; monthly uptime was 99.5%."
}

def lookup_account(customer_id: str) -> str:
    """Return the billing record for a customer."""
    return accounts.get(customer_id, "Customer not found")

agent = dspy.ReAct(
    signature=BillingDecision,
    tools=[lookup_account],
    max_iters=2,
)

result = agent(
    request="Customer C-204 reports an outage and requests a credit.",
    policy=(
        "Approve an outage credit only when the account shows a verified "
        "outage and monthly uptime below 99.9%. Otherwise, deny."
    ),
)

print(result.decision)
print(result.trajectory)
```

In [ ]:
OUTAGE_POLICY = """
For a verified platform outage with monthly uptime below 99.9%, issue a service credit equal to
min(monthly_fee * downtime_minutes / 43,200 * 10, monthly_fee * 0.30), rounded to cents,
and route to reliability. If the required account facts are unavailable, escalate to billing for $0.
""".strip()

ACCOUNTS = {
    "C-204": {
        "monthly_fee": 1000.0,
        "monthly_uptime": 99.5,
        "verified_outage_minutes": 216,
    }
}
TOOL_CALLS = 0


def lookup_account(customer_id: str) -> str:
    """Return the billing and reliability snapshot for a customer ID."""
    global TOOL_CALLS
    TOOL_CALLS += 1
    return str(ACCOUNTS.get(customer_id, "customer not found"))


def calculate_service_credit(monthly_fee: float, downtime_minutes: int) -> float:
    """Calculate the exact service credit dictated by the outage formula."""
    global TOOL_CALLS
    TOOL_CALLS += 1
    return round(
        min(monthly_fee * downtime_minutes / 43_200 * 10, monthly_fee * 0.30),
        2,
    )


class ResolveLiveTicket(dspy.Signature):
    """Resolve a live billing ticket. Use tools for missing account facts and exact arithmetic."""

    ticket: str = dspy.InputField()
    policy: str = dspy.InputField()
    decision: Literal["approve", "deny", "escalate"] = dspy.OutputField()
    amount: float = dspy.OutputField()
    route: Literal["billing", "retention", "reliability", "security"] = dspy.OutputField()
    reply: str = dspy.OutputField()


live_ticket = (
    "Customer C-204 reports an outage on this month's bill and asks for the credit they are owed."
)
live_expected = dspy.Example(
    decision="approve",
    amount=50.0,
    route="reliability",
)

no_tools = dspy.ChainOfThought(ResolveLiveTicket)
no_tools_result = no_tools(ticket=live_ticket, policy=OUTAGE_POLICY)

agent = dspy.ReAct(
    ResolveLiveTicket,
    tools=[lookup_account, calculate_service_credit],
    max_iters=2,
)
TOOL_CALLS = 0
agent_result = agent(ticket=live_ticket, policy=OUTAGE_POLICY)

tool_name_keys = sorted(
    (key for key in agent_result.trajectory if key.startswith("tool_name_")),
    key=lambda key: int(key.rsplit("_", 1)[1]),
)
tool_names = [agent_result.trajectory[key] for key in tool_name_keys]
trajectory_frame = pd.DataFrame([
    {"step": key, "value": value}
    for key, value in agent_result.trajectory.items()
])

print("Without tools:", no_tools_result.decision, no_tools_result.amount, no_tools_result.route)
print("With ReAct:", agent_result.decision, agent_result.amount, agent_result.route)
print("Tool calls:", TOOL_CALLS)
print(trajectory_frame.to_string(index=False))

assert not billing_metric(live_expected, no_tools_result, trace=True)
assert billing_metric(live_expected, agent_result, trace=True)
assert tool_names == ["lookup_account", "calculate_service_credit"]

## 8. Compose multiple Signatures in one Module

Making every ticket an agent is not automatically better. A policy question with all facts in the text should not spend iterations calling account tools; on our small model, doing so can actually make the answer worse.

A real program rarely has one prompt do everything. Here a custom `dspy.Module` fits together **three different Signatures** with one public interface:

1. `RouteTicket` classifies whether the request is self-contained or needs account data.
2. `ResolvePolicyTicket` handles self-contained policy questions without tools.
3. `ResolveLiveTicket` powers a ReAct agent for cases that need retrieval and exact calculation.

This is decomposition in DSPy: each stage has a narrow typed contract, and `forward` wires their predictions together. The module can later be optimized as a unit while each predictor remains independently inspectable and replaceable.

A customer ID alone does not decide the path: a self-contained ticket can include an ID and still route to `policy_only`. The public input and output contract stays the same. Only the internal program changes. The router adds one classifier LM call per ticket, so this comparison does **not** claim fewer total LM calls; it measures decision quality and expensive tool/agent work.

### Define a routing Signature

```python
class RouteBillingRequest(dspy.Signature):
    """Decide whether a billing request requires account data."""

    request: str = dspy.InputField()
    path: Literal["policy_only", "account_lookup"] = dspy.OutputField()
```

### Compose the router and specialists in a custom Module

```python
class BillingCopilot(dspy.Module):
    def __init__(self):
        super().__init__()
        self.router = dspy.Predict(RouteBillingRequest)
        self.policy_only = dspy.ChainOfThought(BillingDecision)
        self.account_lookup = agent

    def forward(self, request, policy):
        route = self.router(request=request)

        if route.path == "account_lookup":
            return self.account_lookup(request=request, policy=policy)

        return self.policy_only(request=request, policy=policy)
```

In [ ]:
PIPELINE_POLICY = """
Annual renewals on days 0-14 are fully refundable only when post-renewal usage is at most 10
compute-hours; otherwise deny. Route renewals to retention. For a verified outage with monthly
uptime below 99.9%, approve a reliability credit of min(monthly_fee * downtime_minutes / 43,200 * 10,
monthly_fee * 0.30), rounded to cents. Missing required account facts means escalate to billing for $0.
""".strip()


class RouteTicket(dspy.Signature):
    """Choose policy_only when all policy-required facts are present; choose account_lookup when required facts must be retrieved."""

    ticket: str = dspy.InputField()
    policy: str = dspy.InputField()
    path: Literal["policy_only", "account_lookup"] = dspy.OutputField()


class ResolvePolicyTicket(dspy.Signature):
    """Resolve a self-contained billing ticket using only explicit facts and policy."""

    ticket: str = dspy.InputField()
    policy: str = dspy.InputField()
    decision: Literal["approve", "deny", "escalate"] = dspy.OutputField()
    amount: float = dspy.OutputField()
    route: Literal["billing", "retention", "reliability", "security"] = dspy.OutputField()
    reply: str = dspy.OutputField()


class RoutedBillingCopilot(dspy.Module):
    def __init__(self):
        super().__init__()
        self.router = dspy.Predict(RouteTicket)
        self.router.set_lm(strong_lm)
        self.router.demos = [
            dspy.Example(
                ticket="I renewed 9 days ago and used 14 compute-hours. Am I eligible?",
                policy=PIPELINE_POLICY,
                path="policy_only",
            ).with_inputs("ticket", "policy"),
            dspy.Example(
                ticket="Customer C-315 renewed 9 days ago for $1,200 and used 14 compute-hours. Refund me.",
                policy=PIPELINE_POLICY,
                path="policy_only",
            ).with_inputs("ticket", "policy"),
            dspy.Example(
                ticket="Customer C-204 had an outage. Look up the account and calculate the credit.",
                policy=PIPELINE_POLICY,
                path="account_lookup",
            ).with_inputs("ticket", "policy"),
        ]
        self.policy_specialist = dspy.ChainOfThought(ResolvePolicyTicket)
        self.account_agent = dspy.ReAct(
            ResolveLiveTicket,
            tools=[lookup_account, calculate_service_credit],
            max_iters=2,
        )

    def forward(self, ticket: str, policy: str):
        route = self.router(ticket=ticket, policy=policy)
        if route.path == "account_lookup":
            result = self.account_agent(ticket=ticket, policy=policy)
        else:
            result = self.policy_specialist(ticket=ticket, policy=policy)
        result.path = route.path
        return result


LIVE_CASES = [
    dspy.Example(
        ticket="I renewed 9 days ago for $1,200 and used 14 compute-hours. Refund me.",
        policy=PIPELINE_POLICY, decision="deny", amount=0.0, route="retention",
    ).with_inputs("ticket", "policy"),
    dspy.Example(
        ticket="Customer C-315 renewed 9 days ago for $1,200 and used 14 compute-hours. Refund me.",
        policy=PIPELINE_POLICY, decision="deny", amount=0.0, route="retention",
    ).with_inputs("ticket", "policy"),
    dspy.Example(
        ticket="Customer C-204 reports an outage on this month's bill and asks for the credit owed.",
        policy=PIPELINE_POLICY, decision="approve", amount=50.0, route="reliability",
    ).with_inputs("ticket", "policy"),
    dspy.Example(
        ticket="Customer C-999 reports an outage and asks for a credit, but no account facts are attached.",
        policy=PIPELINE_POLICY, decision="escalate", amount=0.0, route="billing",
    ).with_inputs("ticket", "policy"),
]
EXPECTED_LIVE_PATHS = [
    "policy_only",
    "policy_only",
    "account_lookup",
    "account_lookup",
]


def evaluate_live(program):
    global TOOL_CALLS
    TOOL_CALLS = 0
    rows = []
    for example in LIVE_CASES:
        pred = program(**example.inputs())
        rows.append({
            "ticket": example.ticket[:48] + "...",
            "path": getattr(pred, "path", "always_agent"),
            "predicted": f"{pred.decision} ${float(pred.amount):.2f} → {pred.route}",
            "score": billing_metric(example, pred),
        })
    frame = pd.DataFrame(rows)
    return frame["score"].mean(), TOOL_CALLS, frame


always_agent = dspy.ReAct(
    ResolveLiveTicket,
    tools=[lookup_account, calculate_service_credit],
    max_iters=2,
)
always_agent_score, always_agent_calls, _ = evaluate_live(always_agent)
routed = RoutedBillingCopilot()
routed_score, routed_calls, routed_rows = evaluate_live(routed)
print("Composed Signatures: RouteTicket → (ResolvePolicyTicket | ResolveLiveTicket)")

comparison = pd.DataFrame([
    {"program": "Always ReAct", "quality": always_agent_score, "tool_calls": always_agent_calls},
    {"program": "Router + specialists", "quality": routed_score, "tool_calls": routed_calls},
])
print(comparison.to_string(index=False))
print(routed_rows.to_string(index=False))

assert routed_rows["path"].tolist() == EXPECTED_LIVE_PATHS
assert routed_score > always_agent_score
assert routed_calls < always_agent_calls

## 9. Scale the same task to long history with an RLM

A tool is ideal when we already know the query we need. Sometimes the evidence is buried in a large, messy account history and the useful search strategy is not known in advance. `dspy.RLM` stores inputs as variables in a sandboxed Python REPL. The model writes code to inspect those variables, can delegate semantic sub-queries, and submits typed outputs when it has enough evidence.

Here we generate 603 account events. Only three lines matter. A normal predictor would receive the entire log as prompt text; the RLM sees previews, writes a filter, calculates the credit in code, and returns the same decision/amount/route contract plus evidence.

### Search long account history with an RLM

```python
class ResolveFromHistory(dspy.Signature):
    """Resolve a billing request using evidence from a long account history."""

    request: str = dspy.InputField()
    account_history: list[str] = dspy.InputField()
    policy: str = dspy.InputField()

    decision: Literal["approve", "deny", "escalate"] = dspy.OutputField()
    evidence: list[str] = dspy.OutputField(
        desc="Relevant facts copied from the account history"
    )


rlm = dspy.RLM(
    ResolveFromHistory,
    sub_lm=small_lm,
    max_iterations=8,
    max_llm_calls=4,
)

result = rlm(
    request="Customer C-204 asks for a credit for the June outage.",
    account_history=account_history,  # 603 account events
    policy=(
        "Approve an outage credit when the outage is verified and monthly "
        "uptime is below 99.9%."
    ),
)

print(result.decision)
print(result.evidence)
```

In [ ]:
if not shutil.which("deno"):
    raise RuntimeError(
        "The RLM section needs Deno. Install it from https://docs.deno.com/runtime/getting_started/installation/"
    )


class ResolveFromHistory(dspy.Signature):
    """Resolve a billing ticket by finding evidence in a long account event history and applying policy."""

    ticket: str = dspy.InputField()
    account_history: list[str] = dspy.InputField()
    policy: str = dspy.InputField()
    decision: Literal["approve", "deny", "escalate"] = dspy.OutputField()
    amount: float = dspy.OutputField()
    route: Literal["billing", "retention", "reliability", "security"] = dspy.OutputField()
    evidence: list[str] = dspy.OutputField(
        desc="Exact relevant facts found in account_history"
    )


noise = [
    f"2026-06-{(i % 28) + 1:02d} customer=C-{300 + i} event=login region=us-east"
    for i in range(600)
]
expected_rlm_evidence = [
    "2026-06-01 customer=C-204 event=plan_snapshot monthly_fee_usd=1000.00",
    "2026-06-18 customer=C-204 event=platform_outage verified=true downtime_minutes=216",
    "2026-06-30 customer=C-204 event=sla_rollup monthly_uptime_percent=99.5",
]
account_history = (
    noise[:180]
    + [expected_rlm_evidence[0]]
    + noise[180:410]
    + [expected_rlm_evidence[1]]
    + noise[410:]
    + [expected_rlm_evidence[2]]
)

rlm = dspy.RLM(
    ResolveFromHistory,
    max_iterations=8,
    max_llm_calls=4,
    max_output_chars=4000,
    sub_lm=small_lm,
)
rlm.set_lm(strong_lm)

rlm_result = rlm(
    ticket="Customer C-204 asks for the service credit owed for the June outage.",
    account_history=account_history,
    policy=OUTAGE_POLICY,
)

print("Events available:", len(account_history))
print("RLM iterations:", len(rlm_result.trajectory))
print("Decision:", rlm_result.decision)
print("Amount:", rlm_result.amount)
print("Route:", rlm_result.route)
print("Evidence:")
for item in rlm_result.evidence:
    print("-", item)
print("\nFirst code action:\n", rlm_result.trajectory[0]["code"])

trajectory_code = "\n".join(
    step.get("code", "")
    for step in rlm_result.trajectory
    if isinstance(step, dict)
)
compact_trajectory_code = trajectory_code.lower().replace("_", "").replace(",", "").replace(" ", "")
rlm_fields_correct = (
    rlm_result.decision == "approve"
    and abs(float(rlm_result.amount) - 50.0) < 0.01
    and rlm_result.route == "reliability"
)
rlm_evidence_exact = rlm_result.evidence == expected_rlm_evidence
rlm_evidence_verbatim = all(item in account_history for item in rlm_result.evidence)
rlm_code_filtered_customer = "customer=C-204" in trajectory_code
rlm_code_applied_policy = "min(" in compact_trajectory_code and "43200" in compact_trajectory_code

assert rlm_fields_correct
assert rlm_evidence_exact
assert rlm_evidence_verbatim
assert rlm_code_filtered_customer
assert rlm_code_applied_policy
rlm_passed = all([
    rlm_fields_correct,
    rlm_evidence_exact,
    rlm_evidence_verbatim,
    rlm_code_filtered_customer,
    rlm_code_applied_policy,
])

## 10. What the progression bought us

The useful DSPy abstraction is not “a nicer prompt.” It is a stable task contract surrounded by interchangeable execution strategies and a feedback loop.

In [ ]:
summary = pd.DataFrame([
    {
        "evaluation_scope": "Main billing dataset",
        "n": len(dataset),
        "comparison": "Predict → ChainOfThought",
        "before": baseline_score,
        "result": cot_score,
        "delta": cot_score - baseline_score,
        "capability_pass": "",
    },
    {
        "evaluation_scope": "Optimizer devset",
        "n": len(optimizer_devset),
        "comparison": "Student → optimized student",
        "before": before_opt_score,
        "result": optimized_score,
        "delta": optimized_score - before_opt_score,
        "capability_pass": "",
    },
    {
        "evaluation_scope": "Mixed live tickets",
        "n": len(LIVE_CASES),
        "comparison": "Always ReAct → router + specialists",
        "before": always_agent_score,
        "result": routed_score,
        "delta": routed_score - always_agent_score,
        "capability_pass": "",
    },
    {
        "evaluation_scope": "Long account history",
        "n": 1,
        "comparison": "RLM capability check",
        "before": None,
        "result": None,
        "delta": None,
        "capability_pass": "PASS" if rlm_passed else "FAIL",
    },
])
print(summary.to_string(index=False))

### A practical selection rule

- Start with `Predict` and an evaluation. Without a baseline, “better” is just a feeling.
- Try `ChainOfThought` when the inputs contain the needed facts but the model applies them badly.
- Compile with an optimizer when failures repeat and you have labeled cases plus a trustworthy metric.
- Use `ReAct` when the answer requires external state or exact operations exposed as tools.
- Decompose with a router when different inputs need different context, cost, or capabilities.
- Use `RLM` when the input itself is too large or irregular for one fixed retrieval step and code can explore it efficiently.

These modules do not replace evaluation. They give you different programs to evaluate against the same operational contract.